In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import pickle

In [2]:
### load the trained model
model = load_model('model.h5')

## load the encoder and scaler and one_hot
with open('label_encoder.pkl','rb') as file:
    label_encoder = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

with open('onehot_encoder.pkl','rb') as file:
    onehot_encoder = pickle.load(file)

In [22]:
#Example input data
input_data ={
    'CreditScore' : 600,
    'Geography' : 'France',
    'Gender' : 'Male',
    'Age' : 40,
    'Tenure': 3,
    'Balance' : 60000,
    'NumOfProducts' : 2,
    'HasCrCard' : 1,
    'IsActiveMember' : 1,
    'EstimatedSalary' : 50000
}

In [24]:
input_data_df = pd.DataFrame([input_data])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [26]:
# one hot encode 'Geography'
geo_encoded = onehot_encoder.transform([input_data_df['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=onehot_encoder.get_feature_names_out(['Geography']))
geo_encoded_df

e:\ANN_Project\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [27]:
# combine
input_data_df = pd.concat([input_data_df.drop('Geography',axis=1),geo_encoded_df],axis=1)
input_data_df


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [28]:
input_data_df['Gender'] = label_encoder.transform(input_data_df['Gender'])
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [ ]:
# input scaled
x_scaled = scaler.transform(input_data_df)

In [31]:
x_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [32]:
# Predict churn
y_pred = model.predict(x_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step


In [33]:
y_pred

array([[0.03384484]], dtype=float32)

In [38]:
prediction_prob = y_pred[0][0]
prediction_prob

np.float32(0.033844844)

In [ ]:
if prediction_prob> 0.5:
    print('the customer is likely to churn.')
else:
    print('the customer is not likely to churn.')

the customer is not likely to churn.
